In [ ]:
import pandas as pd
from typing import Tuple, Optional


In [ ]:
data = "/home/jupyter/workspaces/infectiousdiseasephewas2/data/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results"
results = "/home/jupyter/workspaces/infectiousdiseasephewas2/results/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results"

scratch = "/home/jupyter/workspaces/infectiousdiseasephewas2/scratch/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results"

#!mkdir {scratch}

In [ ]:
#import aux file functions

import pandas as pd

def load_ancestry(path: str = f"{data}/ancestry.csv") -> pd.DataFrame:
    return pd.read_csv(path)

def load_demo(path: str = f"{data}/demographics_table.csv") -> pd.DataFrame:
    return pd.read_csv(path)

def load_flagged(path: str = f"{data}/flagged_samples.tsv") -> pd.DataFrame:
    # assuming tab-separated file
    return pd.read_csv(path, sep="\t")

def load_phex(path: str = f"{data}/phecodex_info.csv") -> pd.DataFrame:
    return pd.read_csv(path)

def load_related(path: str = f"{data}/relatedness_flagged_samples.tsv") -> pd.DataFrame:
    # assuming tab-separated file
    return pd.read_csv(path, sep="\t")

def load_phex_map(path: str = f"{data}/updated_phecodex_map.csv") -> pd.DataFrame:
    return pd.read_csv(path)

def load_srwgs_samples(path: str = f"{data}/v7_has_srWGS.csv") -> pd.DataFrame:
    return pd.read_csv(path)




In [ ]:
#import pcs
def flatten_ancestry_pcs_df(pc_file): 
    
    pc_df = pd.read_csv(pc_file, sep='\t')
    
    pc_df["pca_features"] = pc_df["pca_features"].str[1:-1]
    
    PCs = pc_df["pca_features"].str.split(",", n = 16, expand = True)
    PCs = PCs.astype(float)
    
    pid = pc_df[["research_id"]]
    
    columns= ["PC1","PC2","PC3","PC4","PC5","PC6","PC7","PC8","PC9","PC10","PC11","PC12","PC13","PC14","PC15","PC16"]
    PCs.columns = columns
    
    PCs_final = pd.concat([pid, PCs], axis = 1)
    
    PCs_final.to_csv('wrangled_ancestry_pcs.csv')
    
    return PCs_final

In [ ]:
#import phecode (== ID_054.3 Hepatitis C,== ID_054.2 Hepatitis B, == ID_089.2 viral infections) samples and controls

def get_case_and_control_df(
    csv_path: str,
    phecode: str,
    sex: Optional[str] = None,   # "M" / "F" or None for both
    sex_col: str = "sex",
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    For a single phecode, return:
      - cases_df:    rows where phecode == True
      - controls_df: rows where phecode == False

    Only person_id, that phecode, and sex are read from the big CSV.
    NaN in the phecode column = out of risk set → excluded.
    """

    # Only load the columns we need from disk
    usecols = ["person_id", phecode, sex_col]
    df = pd.read_csv(csv_path, usecols=usecols)

    # valid rows for this phecode
    mask_valid = df[phecode].notna()

    # optional sex restriction
    if sex is not None:
        mask_valid &= df[sex_col] == sex

    # Cases: True
    mask_cases = mask_valid & (df[phecode] == True)

    # Controls: False
    mask_controls = mask_valid & (df[phecode] == False)

    cols_out = ["person_id", sex_col]

    cases_df = df.loc[mask_cases, cols_out].reset_index(drop=True)
    controls_df = df.loc[mask_controls, cols_out].reset_index(drop=True)

    return cases_df, controls_df

In [ ]:
def remove_flagged_and_related_samples(flag_df, related_df, df_case, df_cont):
    # Collect IDs to drop
    related_ids = set(related_df["sample_id"])
    flagged_ids = set(flag_df["s"])
    ids_to_drop = related_ids | flagged_ids  # union

    def _filter(df):
        return df[~df["person_id"].isin(ids_to_drop)].reset_index(drop=True)

    df_final_case = _filter(df_case)
    df_final_cont = _filter(df_cont)

    return df_final_case, df_final_cont

In [ ]:
#merge filtered case and control files with demo and ancestry data

def add_demo_and_ancesry_data(demo, ancestry,  fil_df_case, fil_df_cont):
    

    demo_case = pd.merge(fil_df_case, demo, on = "person_id", how = "left")
    final_case = pd.merge(demo_case, ancestry, on = "person_id", how = "left")
    
    demo_cont = pd.merge(fil_df_cont, demo, on = "person_id", how = "left")
    final_cont = pd.merge(demo_cont, ancestry, on = "person_id", how = "left")
    
    return final_case, final_cont
    

    


In [ ]:
def add_pcs_to_covar_files_and_filter_for_srwgs(pcs, srwgs, case, control):
    
    pcs = pcs.rename(columns={"research_id": "person_id"})
    
    case_pcs = pd.merge(case, pcs, on = "person_id", how = "left")
    cont_pcs = pd.merge(control, pcs, on = "person_id", how = "left")
    
    sr_samples = srwgs["person_id"]
    
    final_case = case_pcs[case_pcs["person_id"].isin(sr_samples)].reset_index(drop=True)
    final_cont = cont_pcs[cont_pcs["person_id"].isin(sr_samples)].reset_index(drop=True)
    
    return final_case, final_cont


In [ ]:
def add_phenotype_col_and_filter_on_sex_and_age(case, control):
    
    
    #cases == 1
    #control == 0
    
    case["case"] = 1
    control["case"] = 0
    
    #filter on age <=100
    
    case = case[case["age_at_cdr"] <= 100]
    control = control[control["age_at_cdr"] <= 100]
    
    #remove samples who dont have sex
    
    sex = ["M", "F"]
   
    final_case = case[case["sex"].isin(sex)].reset_index(drop=True)
    final_cont = control[control["sex"].isin(sex)].reset_index(drop=True)
    
    return final_case, final_cont
    
    

In [ ]:
def get_missing_covar_cols (case, control):
    
    case = case.copy()
    control = control.copy()
    
    
    #convert sex to 0 = male, and 1 = female
   
    case["sex"] = case["sex"].map({"M": 0, "F": 1})
    control["sex"] = control["sex"].map({"M": 0, "F": 1})

    #add age*sex, age^2, age^2*sex, 
    
    case["age2"] = case["age_at_cdr"]**2
    control["age2"] = control["age_at_cdr"]**2

    case["age_sex"] = case["sex"] * case["age_at_cdr"]
    control["age_sex"] = control["sex"] * control["age_at_cdr"]
    
    case["age2_sex"] = case["sex"] * case["age2"]
    control["age2_sex"] = control["sex"] * control["age2"]
    
    
    return case, control 

In [ ]:
def get_concat_pheno_covar_files_for_case_controls(case, control, filename):
    
    final = pd.concat([case, control], axis = 0,  ignore_index=True)
    
    
    final = final.rename(columns={"age_at_cdr": "age"})
    
    final.to_csv(filename)
    
    return final

In [ ]:
#function calls
'''
ancestry = load_ancestry()
demo = load_demo()
flagged = load_flagged()
phex = load_phex()
related = load_related()
phex_map = load_phex_map()
pcs = flatten_ancestry_pcs_df(f"{data}/ancestry_preds.tsv")
srwgs = load_srwgs_samples()

hepc, hepc_controls  = get_case_and_control_df(f"{data}/mcc2_phecodex_table.csv", "ID_054.3")
hepb, hepb_controls   = get_case_and_control_df(f"{data}/mcc2_phecodex_table.csv", "ID_054.2")
viral, viral_controls = get_case_and_control_df(f"{data}/mcc2_phecodex_table.csv", "ID_089.2")


hepc_fil_case, hepc_fil_cont = remove_flagged_and_related_samples(flagged, related, hepc, hepc_controls)
hepb_fil_case, hepb_fil_cont = remove_flagged_and_related_samples(flagged, related, hepb, hepb_controls)
viral_fil_case, viral_fil_cont = remove_flagged_and_related_samples(flagged, related, viral, viral_controls)


hepc_df, hepc_cont_df = add_demo_and_ancesry_data(demo, ancestry, hepc_fil_case, hepc_fil_cont)
hepb_df, hepb_cont_df = add_demo_and_ancesry_data(demo, ancestry, hepb_fil_case, hepb_fil_cont)
viral_df, viral_cont_df = add_demo_and_ancesry_data(demo, ancestry, viral_fil_case, viral_fil_cont)


hepc_phenos, hepc_cont_phenos = add_pcs_to_covar_files_and_filter_for_srwgs(pcs,srwgs, hepc_df, hepc_cont_df)
hepb_phenos, hepb_cont_phenos = add_pcs_to_covar_files_and_filter_for_srwgs(pcs,srwgs, hepb_df, hepb_cont_df)
viral_phenos, viral_cont_phenos = add_pcs_to_covar_files_and_filter_for_srwgs(pcs, srwgs, viral_df, viral_cont_df)


hepc2, hepc2_control = add_phenotype_col_and_filter_on_sex_and_age(hepc_phenos, hepc_cont_phenos)
hepb2, hepb2_control = add_phenotype_col_and_filter_on_sex_and_age(hepb_phenos, hepb_cont_phenos)
viral2, viral2_cont = add_phenotype_col_and_filter_on_sex_and_age(viral_phenos, viral_cont_phenos)


hepc3, hepc3_control = get_missing_covar_cols (hepc2, hepc2_control)
hepb3, hepb3_control = get_missing_covar_cols (hepb2, hepb2_control)
viral3, viral3_cont = get_missing_covar_cols (viral2, viral2_cont)
'''


final_hepc = get_concat_pheno_covar_files_for_case_controls(hepc3, hepc3_control, "hepc_matchit.csv")
final_hepb = get_concat_pheno_covar_files_for_case_controls(hepb3, hepb3_control, "hepb_matchit.csv")
final_viral = get_concat_pheno_covar_files_for_case_controls(viral3, viral3_cont, "viral_matchit.csv")



In [ ]:
#visualize

#ancestry 
#demo
#flagged
#phex 
#related 
#phex_map
#pcs
#hepc
#hepc_controls 
#hepb
#hepb_controls   
#viral
#viral_controls

#demo

#hepc_fil_case
#hepc_fil_cont

#hepb_fil_case 
#hepb_fil_cont 

#viral_fil_case
#viral_fil_cont


#hepc_df
#hepc_cont_df

#hepb_df
#hepb_cont_df 

#viral_df
#viral_cont_df


#hepc_phenos
#hepc_cont_phenos 

#hepb_phenos
#hepb_cont_phenos 

viral_phenos
viral_cont_phenos 


#hepc2
#hepc2_control

#hepb2
#hepb2_control

viral2
#viral2_cont


hepc3
hepc3_control 

hepb3
hepb3_control 

viral3
viral3_cont 


In [ ]:
# Number of missing values per column
missing_counts = viral3_cont.isna().sum()

# Column with the most missing values
most_missing_col = missing_counts.idxmax()
most_missing_count = missing_counts.max()

print("Column with most missing values:", most_missing_col)
print("Number of missing values:", most_missing_count)

# If you want to see all columns sorted by missingness:
print(missing_counts.sort_values(ascending=False))

In [ ]:
df5 = final_hepc[final_hepc["case"]== 1]

In [ ]:
df5

In [ ]:
pwd

In [ ]:
#transfer to other workspace to run regenie

#cp plink files and mt to new workspace from old 
!gsutil -m cp  /home/jupyter/workspaces/infectiousdiseasephewas2/results/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results/hepb_matched_10.csv   gs://fc-secure-d55f68ec-0246-4975-8f6e-0ccd7db6a76b/notebooks/results/  


In [ ]:
#transfer to other workspace to run regenie

#cp plink files and mt to new workspace from old 
!gsutil -m cp  /home/jupyter/workspaces/infectiousdiseasephewas2/results/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results/hepb_matched_ALL.csv   gs://fc-secure-d55f68ec-0246-4975-8f6e-0ccd7db6a76b/notebooks/results/  


In [ ]:
#transfer to other workspace to run regenie

#cp plink files and mt to new workspace from old 
!gsutil -m cp  /home/jupyter/workspaces/infectiousdiseasephewas2/results/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results/hepc_matched_ALL.csv   gs://fc-secure-d55f68ec-0246-4975-8f6e-0ccd7db6a76b/notebooks/results/  


In [ ]:
#transfer to other workspace to run regenie

#cp plink files and mt to new workspace from old 
!gsutil -m cp  /home/jupyter/workspaces/infectiousdiseasephewas2/results/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results/hepc_matched_10.csv   gs://fc-secure-d55f68ec-0246-4975-8f6e-0ccd7db6a76b/notebooks/results/  


In [ ]:
#transfer to other workspace to run regenie

#cp plink files and mt to new workspace from old 
!gsutil -m cp  /home/jupyter/workspaces/infectiousdiseasephewas2/results/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results/viral_matched_10.csv   gs://fc-secure-d55f68ec-0246-4975-8f6e-0ccd7db6a76b/notebooks/results/  


In [ ]:
#transfer to other workspace to run regenie

#cp plink files and mt to new workspace from old 
!gsutil -m cp  /home/jupyter/workspaces/infectiousdiseasephewas2/results/2026-02-18_validate_regenie_pipeline_by_replicating_AoU_gwas_results/viral_matched_ALL.csv   gs://fc-secure-d55f68ec-0246-4975-8f6e-0ccd7db6a76b/notebooks/results/  
